In [ ]:
# Load dataset
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler


df = pd.read_csv('dataset.csv')

# Prepare target and features (ignore "segment" column)
feature_cols = [col for col in df.columns if col not in ['segment', 'anomaly', 'train', 'channel']]
X = df[feature_cols].values
y = df['anomaly'].values
train_mask = df['train'].values == 1
test_mask = df['train'].values == 0

# Scale features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Reshape for CNN input (samples, timesteps, features)
X = np.expand_dims(X, axis=2)

# Prepare train and test splits
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# If anomaly is not categorical, make to_categorical (otherwise skip)
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)


NameError: name 'pd' is not defined

In [ ]:
# Build simple 1D CNN model
model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    Dropout(0.2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(y_train.shape[1], activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.3f}")


# Adversarial Attack
Here is a Python code example for running adversarial attacks on a machine learning model using the Adversarial Robustness Toolbox (ART), specifically demonstrating a Fast Gradient Method (FGM) attack. You can modify this code for other attacks such as PGD or DeepFool easily.



In [ ]:
from art.attacks.evasion import FastGradientMethod
from art.estimators.classification import TensorFlowV2Classifier
import tensorflow as tf


In [1]:
# Wrap model with ART classifier
classifier = TensorFlowV2Classifier(
    model=model,
    nb_classes=y_train.shape[1],
    input_shape=(X_train.shape[1], 1),
    loss_object=tf.keras.losses.CategoricalCrossentropy()
)

# Run Fast Gradient Method attack
attack = FastGradientMethod(estimator=classifier, eps=0.2)
X_test_adv = attack.generate(x=X_test)

# Evaluate on adversarial samples
preds = np.argmax(classifier.predict(X_test_adv), axis=1)
acc = np.sum(preds == np.argmax(y_test, axis=1)) / len(y_test)
print("Test accuracy on adversarial examples: {:.2f}%".format(acc * 100))

NameError: name 'TensorFlowV2Classifier' is not defined

In [ ]:
# PGD Attack
from art.attacks.evasion import ProjectedGradientDescent

pgd_attack = ProjectedGradientDescent(
    estimator=classifier,
    norm=2,               # or infinity ('inf')
    eps=0.3,
    eps_step=0.1,
    max_iter=100,
    targeted=False
)
X_test_pgd = pgd_attack.generate(x=X_test)
preds_pgd = np.argmax(classifier.predict(X_test_pgd), axis=1)
acc_pgd = np.sum(preds_pgd == np.argmax(y_test, axis=1)) / len(y_test)
print("Test accuracy on PGD adversarial examples: {:.2f}%".format(acc_pgd * 100))

In [ ]:
from art.attacks.evasion import DeepFool
# DeepFool Attack
deepfool_attack = DeepFool(
    classifier,
    max_iter=50,
    epsilon=1e-6,
    nb_grads=10,
    batch_size=32
)
X_test_deepfool = deepfool_attack.generate(x=X_test)
preds_deepfool = np.argmax(classifier.predict(X_test_deepfool), axis=1)
acc_deepfool = np.sum(preds_deepfool == np.argmax(y_test, axis=1)) / len(y_test)
print("Test accuracy on DeepFool adversarial examples: {:.2f}%".format(acc_deepfool * 100))

# 
Here are example snippets for several additional attacks available in the Adversarial Robustness Toolbox (ART). You can run these attacks in a similar fashion to PGD/DeepFool after your ART classifier setup:



In [ ]:
from art.attacks.evasion import HopSkipJump

# HopSkipJump attack (for classifiers with decision function)
hsj_attack = HopSkipJump(classifier=classifier, max_iter=10, max_eval=100, init_eval=10)
X_test_hsj = hsj_attack.generate(x=X_test)
preds_hsj = np.argmax(classifier.predict(X_test_hsj), axis=1)
acc_hsj = np.sum(preds_hsj == np.argmax(y_test, axis=1)) / len(y_test)
print("Test accuracy on HopSkipJump adversarial examples: {:.2f}%".format(acc_hsj * 100))


In [3]:
from art.attacks.evasion import ElasticNet

# ElasticNet Attack
enet_attack = ElasticNet(classifier=classifier, beta=1.0, max_iter=10)
X_test_enet = enet_attack.generate(x=X_test)
preds_enet = np.argmax(classifier.predict(X_test_enet), axis=1)
acc_enet = np.sum(preds_enet == np.argmax(y_test, axis=1)) / len(y_test)
print("Test accuracy on ElasticNet adversarial examples: {:.2f}%".format(acc_enet * 100))



ModuleNotFoundError: No module named 'art'